In [66]:
from pathlib import Path    
import librosa
import os
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pylab
import sklearn
import tensorflow as tf
from tensorflow.keras import layers, models,optimizers
import os
from PIL import Image
import numpy as np
import tensorflow as tf 
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [67]:
# Define the path to the dataset
main_dir = 'multilabel/'

# Get the list of subfolders (classes)
classes = os.listdir(main_dir)
class_indices = {class_name: index for index, class_name in enumerate(classes)}

# Initialize lists to store images and labels
images = []
labels = []

# Loop through each class folder
for class_name in classes:
    class_folder = os.path.join(main_dir, class_name)
    for file_name in os.listdir(class_folder):
        file_path = os.path.join(class_folder, file_name)
        if file_name.endswith('.png') or file_name.endswith('.jpg') or file_name.endswith('.jpeg'):
            image = Image.open(file_path)
            image = image.resize((32, 32))  # Resize images for consistency
            image = np.array(image)
            images.append(image)
            labels.append(class_indices[class_name])

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

# Normalize images
images = images / 255.0

# Convert labels to categorical
labels = to_categorical(labels, num_classes=len(classes))

# Split the dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=5)


In [85]:
import tensorflow as tf
from tensorflow.keras import layers, models

class DiamondLayer(layers.Layer):
    def __init__(self, diamond_size, depth, **kwargs):
        super(DiamondLayer, self).__init__(**kwargs)
        self.diamond_size = diamond_size
        self.depth = depth

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name='kernel',
            shape=(self.diamond_size, self.diamond_size, input_shape[-1], self.depth),
            initializer='glorot_uniform',
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.depth,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        input_shape = tf.shape(inputs)

        if len(inputs.shape) == 4:
            input_height = input_shape[1]
            input_width = input_shape[2]
            input_channels = input_shape[3]
        elif len(inputs.shape) == 3:
            input_height = input_shape[1]
            input_width = input_shape[2]
            input_channels = 1
            inputs = tf.expand_dims(inputs, axis=-1)
        else:
            raise ValueError("Unsupported input shape: {}".format(inputs.shape))

        pad_height = (self.diamond_size - input_height % self.diamond_size) % self.diamond_size
        pad_width = (self.diamond_size - input_width % self.diamond_size) % self.diamond_size
        padded_input = tf.pad(inputs, [[0, 0], [0, pad_height], [0, pad_width], [0, 0]])

        patches = tf.image.extract_patches(
            images=padded_input,
            sizes=[1, self.diamond_size, self.diamond_size, 1],
            strides=[1, self.diamond_size, self.diamond_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )

        patches = tf.reshape(
            patches,
            [batch_size, -1, self.diamond_size, self.diamond_size, input_channels]
        )

        patches = tf.expand_dims(patches, axis=-1)
        kernel = tf.expand_dims(self.kernel, axis=0)
        kernel = tf.expand_dims(kernel, axis=0)

        output = tf.reduce_sum(
            patches * kernel,
            axis=[2, 3, 4]
        )

        output = tf.nn.relu(output + self.bias)

        # Ensure the output shape is compatible with the next layer
        output_shape = tf.cast(tf.math.sqrt(tf.cast(tf.shape(output)[1], tf.float32)), tf.int32)
        output = tf.reshape(output, [batch_size, output_shape, output_shape, self.depth])

        return output

    def get_config(self):
        config = super(DiamondLayer, self).get_config()
        config.update({'diamond_size': self.diamond_size, 'depth': self.depth})
        return config

# Define the Diamond Neural Network with Weight Sharing
def create_diamond_network(input_shape, diamond_size, depth, num_diamond_layers):
    inputs = layers.Input(shape=input_shape)

    x = inputs
    for _ in range(num_diamond_layers):
        x = DiamondLayer(diamond_size, depth)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)

    # Add a global average pooling layer
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(5, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model


In [86]:
# Print data shapes for verification
print("Shape of x_train:", x_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_test:", y_test.shape)


# Create the model
model = create_diamond_network((32, 32, 4), diamond_size=25, depth=24, num_diamond_layers=3)

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()


Shape of x_train: (572, 32, 32, 4)
Shape of y_train: (572, 5)
Shape of x_test: (143, 32, 32, 4)
Shape of y_test: (143, 5)
Model: "model_22"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_39 (InputLayer)       [(None, 32, 32, 4)]       0         
                                                                 
 diamond_layer_92 (DiamondLa  (None, None, None, 24)   60024     
 yer)                                                            
                                                                 
 batch_normalization_79 (Bat  (None, None, None, 24)   96        
 chNormalization)                                                
                                                                 
 dropout_101 (Dropout)       (None, None, None, 24)    0         
                                                                 
 diamond_layer_93 (DiamondLa  (None, None, None, 24)   360024    
 y

In [87]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



In [88]:

# Train the model
model.fit(x_train, y_train, epochs=5, batch_size=5, validation_data=(x_test, y_test))




Epoch 1/5
115/115 [==============================] - 6s 36ms/step - loss: 1.6687 - accuracy: 0.2448 - val_loss: 1.6039 - val_accuracy: 0.2657
Epoch 2/5
115/115 [==============================] - 4s 33ms/step - loss: 1.6478 - accuracy: 0.2290 - val_loss: 1.5999 - val_accuracy: 0.2657
Epoch 3/5
115/115 [==============================] - 4s 31ms/step - loss: 1.6102 - accuracy: 0.2745 - val_loss: 1.5752 - val_accuracy: 0.3007
Epoch 4/5
115/115 [==============================] - 4s 32ms/step - loss: 1.5755 - accuracy: 0.3007 - val_loss: 1.4927 - val_accuracy: 0.3007
Epoch 5/5
115/115 [==============================] - 3s 29ms/step - loss: 1.5076 - accuracy: 0.3199 - val_loss: 1.4123 - val_accuracy: 0.5035


In [89]:

# Ensure the output shape is correct during evaluation
eval_result = model.evaluate(x_test, y_test, batch_size=batch_size)
print("Test loss:", eval_result[0])
print("Test accuracy:", eval_result[1])

5/5 [==============================] - 0s 52ms/step - loss: 1.4123 - accuracy: 0.5035
Test loss: 1.4122720956802368
Test accuracy: 0.503496527671814


In [102]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

class DiamondLayer(layers.Layer):
    def __init__(self, diamond_size, depth, **kwargs):
        super(DiamondLayer, self).__init__(**kwargs)
        self.diamond_size = diamond_size
        self.depth = depth

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name='kernel',
            shape=(self.diamond_size, self.diamond_size, input_shape[-1], self.depth),
            initializer='glorot_uniform',
            regularizer=regularizers.l2(0.01),  # L2 regularization
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.depth,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        input_shape = tf.shape(inputs)

        if len(inputs.shape) == 4:
            input_height = input_shape[1]
            input_width = input_shape[2]
            input_channels = input_shape[3]
        elif len(inputs.shape) == 3:
            input_height = input_shape[1]
            input_width = input_shape[2]
            input_channels = 1
            inputs = tf.expand_dims(inputs, axis=-1)
        else:
            raise ValueError("Unsupported input shape: {}".format(inputs.shape))

        pad_height = (self.diamond_size - input_height % self.diamond_size) % self.diamond_size
        pad_width = (self.diamond_size - input_width % self.diamond_size) % self.diamond_size
        padded_input = tf.pad(inputs, [[0, 0], [0, pad_height], [0, pad_width], [0, 0]])

        patches = tf.image.extract_patches(
            images=padded_input,
            sizes=[1, self.diamond_size, self.diamond_size, 1],
            strides=[1, self.diamond_size, self.diamond_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )

        patches = tf.reshape(
            patches,
            [batch_size, -1, self.diamond_size, self.diamond_size, input_channels]
        )

        patches = tf.expand_dims(patches, axis=-1)
        kernel = tf.expand_dims(self.kernel, axis=0)
        kernel = tf.expand_dims(kernel, axis=0)

        output = tf.reduce_sum(
            patches * kernel,
            axis=[2, 3, 4]
        )

        output = tf.nn.relu(output + self.bias)

        # Calculate the output shape properly
        num_patches = tf.shape(output)[1]
        output_height = tf.cast(tf.math.sqrt(tf.cast(num_patches, tf.float32)), tf.int32)
        output_width = output_height
        output = tf.reshape(output, [batch_size, output_height, output_width, self.depth])

        return output

    def get_config(self):
        config = super(DiamondLayer, self).get_config()
        config.update({'diamond_size': self.diamond_size, 'depth': self.depth})
        return config


def create_diamond_network(input_shape, diamond_size, depth, num_diamond_layers):
    inputs = layers.Input(shape=input_shape)

    x = inputs
    for _ in range(num_diamond_layers):
        x = DiamondLayer(diamond_size, depth)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.6)(x)  # Increased dropout rate

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.6)(x)  # Increased dropout rate
    outputs = layers.Dense(5, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model


# Create the updated model
model = create_diamond_network((32, 32, 4), diamond_size=5, depth=8, num_diamond_layers=3)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Define the learning rate scheduler callback
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Train the model with the learning rate scheduler
model.fit(x_train, y_train, epochs=10, batch_size=10, validation_data=(x_test, y_test), callbacks=[lr_scheduler])

# Evaluate the model
eval_result = model.evaluate(x_test, y_test, batch_size=5)
print("Test loss:", eval_result[0])
print("Test accuracy:", eval_result[1])


Epoch 1/100
58/58 [==============================] - 2s 13ms/step - loss: 1.9614 - accuracy: 0.2080 - val_loss: 1.7561 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 2/100
58/58 [==============================] - 0s 7ms/step - loss: 1.8898 - accuracy: 0.2150 - val_loss: 1.7366 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 3/100
58/58 [==============================] - 1s 12ms/step - loss: 1.8467 - accuracy: 0.2238 - val_loss: 1.7332 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 4/100
58/58 [==============================] - 1s 11ms/step - loss: 1.8079 - accuracy: 0.2220 - val_loss: 1.7213 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 5/100
58/58 [==============================] - 1s 10ms/step - loss: 1.8397 - accuracy: 0.2255 - val_loss: 1.7195 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 6/100
58/58 [==============================] - 1s 12ms/step - loss: 1.8090 - accuracy: 0.2185 - val_loss: 1.7055 - val_accuracy: 0.2657 - lr: 0.0010
Epoch 7/100
58/58 [==============================] - 1s 10ms/step - los

58/58 [==============================] - 1s 11ms/step - loss: 1.2110 - accuracy: 0.4510 - val_loss: 1.0543 - val_accuracy: 0.4266 - lr: 2.5000e-04
Epoch 54/100
58/58 [==============================] - 0s 8ms/step - loss: 1.2015 - accuracy: 0.4563 - val_loss: 1.0489 - val_accuracy: 0.4266 - lr: 2.5000e-04
Epoch 55/100
58/58 [==============================] - 0s 7ms/step - loss: 1.1658 - accuracy: 0.4685 - val_loss: 1.0448 - val_accuracy: 0.4266 - lr: 2.5000e-04
Epoch 56/100
58/58 [==============================] - 1s 9ms/step - loss: 1.1290 - accuracy: 0.4650 - val_loss: 1.0453 - val_accuracy: 0.4266 - lr: 1.2500e-04
Epoch 57/100
58/58 [==============================] - 1s 9ms/step - loss: 1.1223 - accuracy: 0.4441 - val_loss: 1.0450 - val_accuracy: 0.4266 - lr: 1.2500e-04
Epoch 58/100
58/58 [==============================] - 1s 10ms/step - loss: 1.1307 - accuracy: 0.4738 - val_loss: 1.0444 - val_accuracy: 0.4266 - lr: 1.2500e-04
Epoch 59/100
58/58 [==============================] - 1s 

KeyboardInterrupt: 